# Markov Decision Processes and Value Iteration

This notebook introduces:
1. **Markov Decision Process** (MDP) formalism
2. A **GridWorld** environment from scratch
3. **Value Iteration** to compute the optimal policy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

%matplotlib inline
plt.rcParams['figure.figsize'] = (6, 6)

## 1. MDP Formalism

An MDP is defined by the tuple $(S, A, P, R, \gamma)$:
- $S$: set of states
- $A$: set of actions
- $P(s' | s, a)$: transition probabilities
- $R(s, a, s')$: reward function
- $\gamma \in [0, 1)$: discount factor

The **Bellman optimality equation**:
$$V^*(s) = \max_a \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma V^*(s') \right]$$

In [ ]:
class GridWorld:
    """Simple 4x4 GridWorld with terminal states and obstacles."""
    
    ACTIONS = {'up': (-1, 0), 'down': (1, 0), 'left': (0, -1), 'right': (0, 1)}
    ACTION_NAMES = list(ACTIONS.keys())
    
    def __init__(self, size=4, goal=(0, 3), trap=(1, 3), obstacles=None):
        self.size = size
        self.goal = goal      # +1 reward
        self.trap = trap      # -1 reward
        self.obstacles = obstacles or [(1, 1)]
        self.terminals = {goal, trap}
    
    def is_valid(self, s):
        r, c = s
        return 0 <= r < self.size and 0 <= c < self.size and s not in self.obstacles
    
    def step(self, state, action):
        """Deterministic transition. Returns (next_state, reward, done)."""
        if state in self.terminals:
            return state, 0.0, True
        dr, dc = self.ACTIONS[action]
        next_s = (state[0] + dr, state[1] + dc)
        if not self.is_valid(next_s):
            next_s = state  # stay in place
        reward = 1.0 if next_s == self.goal else (-1.0 if next_s == self.trap else -0.04)
        done = next_s in self.terminals
        return next_s, reward, done
    
    def all_states(self):
        return [(r, c) for r in range(self.size) for c in range(self.size)
                if (r, c) not in self.obstacles]

env = GridWorld()
print(f"States: {len(env.all_states())}, Actions: {len(env.ACTION_NAMES)}")
print(f"Goal: {env.goal}, Trap: {env.trap}, Obstacles: {env.obstacles}")

## 2. Value Iteration

Iteratively apply the Bellman update until $V$ converges:
$$V_{k+1}(s) = \max_a \sum_{s'} P(s'|s,a)[R(s,a,s') + \gamma V_k(s')]$$

In [ ]:
def value_iteration(env, gamma=0.99, theta=1e-6, max_iter=1000):
    V = {s: 0.0 for s in env.all_states()}
    
    for iteration in range(max_iter):
        delta = 0
        for s in env.all_states():
            if s in env.terminals:
                continue
            old_v = V[s]
            action_values = []
            for a in env.ACTION_NAMES:
                s_next, r, _ = env.step(s, a)
                action_values.append(r + gamma * V[s_next])
            V[s] = max(action_values)
            delta = max(delta, abs(V[s] - old_v))
        if delta < theta:
            print(f"Converged in {iteration+1} iterations (delta={delta:.2e})")
            break
    
    # Extract policy
    policy = {}
    for s in env.all_states():
        if s in env.terminals:
            policy[s] = 'X'
            continue
        best_a, best_v = None, -np.inf
        for a in env.ACTION_NAMES:
            s_next, r, _ = env.step(s, a)
            v = r + gamma * V[s_next]
            if v > best_v:
                best_v = v
                best_a = a
        policy[s] = best_a
    return V, policy

V, policy = value_iteration(env)

In [ ]:
# Visualise value function and policy
arrow_map = {'up': '\u2191', 'down': '\u2193', 'left': '\u2190', 'right': '\u2192', 'X': 'X'}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Value function heatmap
V_grid = np.full((env.size, env.size), np.nan)
for s in env.all_states():
    V_grid[s] = V[s]
axes[0].imshow(V_grid, cmap='RdYlGn', interpolation='nearest')
for s in env.all_states():
    axes[0].text(s[1], s[0], f"{V[s]:.2f}", ha='center', va='center', fontsize=12)
axes[0].set_title('Value Function V*(s)')

# Policy
axes[1].imshow(V_grid, cmap='RdYlGn', interpolation='nearest', alpha=0.3)
for s in env.all_states():
    axes[1].text(s[1], s[0], arrow_map[policy[s]], ha='center', va='center', fontsize=20)
# Mark obstacle
for obs in env.obstacles:
    axes[1].text(obs[1], obs[0], '\u2588', ha='center', va='center', fontsize=20, color='black')
axes[1].set_title('Optimal Policy')

for ax in axes:
    ax.set_xticks(range(env.size))
    ax.set_yticks(range(env.size))
plt.tight_layout()
plt.show()

In [ ]:
# Simulate an episode following the optimal policy
state = (3, 0)  # start bottom-left
trajectory = [state]
total_reward = 0

for _ in range(20):
    if state in env.terminals:
        break
    action = policy[state]
    state, reward, done = env.step(state, action)
    trajectory.append(state)
    total_reward += reward

print(f"Trajectory: {' -> '.join(str(s) for s in trajectory)}")
print(f"Total reward: {total_reward:.2f}")

## Key Takeaways

- An **MDP** formalises sequential decision-making under uncertainty.
- **Value iteration** finds the optimal value function and policy via dynamic programming.
- It requires a complete model of the environment ($P$ and $R$) -- this is the **model-based** setting.

**Next:** Q-Learning -- a model-free approach that learns from experience.